# **Importing dependencies**

In [1]:
import warnings
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import talib
import requests
from datetime import datetime, timedelta
from transformers import pipeline
import torch
from huggingface_hub import InferenceClient
import textwrap
import time
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline as skpipe
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, make_scorer
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import xgboost as xgb
from sklearn.preprocessing import MinMaxScaler
warnings.filterwarnings('ignore')

# if kernel doesnt have downloaded yfinance and talib run code below
# !pip install yfinance
# !pip install TA-Lib
# !pip install transformers
# !pip install torch 
# !pip install xgboost

# **Importing stock data from yahoo finance**

In [2]:
stock_tickers = ['PCAR', 'AMAT', 'MCHP', 'CEG']

stock_data = yf.download(tickers=stock_tickers, period='3y', interval='1d', group_by='ticker')

[*********************100%***********************]  4 of 4 completed


data separation

In [3]:
# serparating the dataframes based on stock

mchp = stock_data['MCHP']
amat = stock_data['AMAT']
pcar = stock_data['PCAR']
ceg = stock_data['CEG']

# **Feature engineering**

Adding technical indicators as features

In [4]:
dfs = [mchp, amat, pcar, ceg]

# loop to compute all technical indicators

for df in dfs:

    # Momentum and trend indicators 
    ema_12 = df['Close'].ewm(span=12, adjust=False).mean()
    ema_26 = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD'] = ema_12 - ema_26

    df['RSI_14'] = talib.RSI(df['Close'])

    df['SD'] = df['Close'].rolling(window=20).std()

    # Return and volume centric indicators
    df['Price_Momentum'] = df['Close'].pct_change()

    df['Vol_Shock']      = df['Volume'].pct_change()

    # Normalised price features
    df['Price_Range'] = (df['High'] - df['Low']) / df['Close']

    ema_20_temp = df['Close'].ewm(span=20, adjust=False).mean()
    df['Dist_from_EMA_20'] = (df['Close'] - ema_20_temp) / ema_20_temp

    # Lagged features to avoid look ahead bias
    df['Close_Lag1'] = df['Close'].shift(1)   
    df['Close_Lag3'] = df['Close'].shift(3)  
    df['Close_Lag5'] = df['Close'].shift(5) 
    df['High_Lag1']  = df['High'].shift(1)   
    df['Low_Lag1']   = df['Low'].shift(1)   

    df['Return_1d']  = df['Close'].pct_change(1)  
    df['Return_5d']  = df['Close'].pct_change(5)  

Data cleaning

In [5]:
# Null values + imputation
names = ["mchp", "amat", "pcar", "ceg"]

for name, df in zip(names, dfs):
    null_count = df.isnull().sum().sum()
    print(f"{name}: {null_count} missing values")
    
# Filling all null values from technical indicators
for df in dfs:

    # Identify numeric columns (int, float)
    numeric_cols = df.select_dtypes(include=['number']).columns
    
    # Fill only those columns using ffil
    df[numeric_cols] = df[numeric_cols].ffill()
    
    # Dropping null values
    df.dropna(inplace=True)

mchp: 52 missing values
amat: 52 missing values
pcar: 52 missing values
ceg: 52 missing values


In [6]:
# confirming imputation
for name, df in zip(names, dfs):
    null_count = df.isnull().sum().sum()
    print(f"{name}: {null_count} missing values")
    
# no null alues in all 4 dataframes

mchp: 0 missing values
amat: 0 missing values
pcar: 0 missing values
ceg: 0 missing values


# **Sentiment analysis for fundamental analysis setup**

In [ ]:
# defining api key
finnhub_api = "YOUR_FINNHUB_API_KEY"

# Define a 1 year window
end_date = datetime.today().strftime('%Y-%m-%d')
start_date = (datetime.today() - timedelta(weeks = 52)).strftime('%Y-%m-%d')

all_news_data = []

# Loop through each stock and pull the news
for ticker in ['MCHP', 'AMAT', 'PCAR', 'CEG']:
    print(f"Fetching news for {ticker}...")
    url = f"https://finnhub.io/api/v1/company-news?symbol={ticker}&from={start_date}&to={end_date}&token={finnhub_api}"
    response = requests.get(url)
    
    if response.status_code == 200:
        news_items = response.json()
        for item in news_items:
            all_news_data.append({
                "Ticker": ticker,
                "Date": datetime.fromtimestamp(item['datetime']).strftime('%Y-%m-%d'),
                "Text": item['headline'] + ". " + item['summary'] 
            })
    else:
        print(f"Failed to fetch {ticker}. Status Code: {response.status_code}")

# Convert to DataFrame
news_df = pd.DataFrame(all_news_data)
print(f"Total articles extracted: {len(news_df)}")


Fetching news for MCHP...
Fetching news for AMAT...
Fetching news for PCAR...
Fetching news for CEG...
Total articles extracted: 928


Extracting fundamental analysis articles

In [8]:
# creating classification pipeline for LLM
finbert = pipeline("text-classification", model="ProsusAI/finbert", top_k=None, truncation=True, max_length=512)

def compute_net_sentiment(text):

    """Passes text to FinBERT and calculates: Positive Probability - Negative Probability"""
    try:

        # returning the first element which is its classification
        result = finbert(text)[0] 
        
        # Extracting individual probabilities
        pos_score = next(item['score'] for item in result if item['label'] == 'positive')
        neg_score = next(item['score'] for item in result if item['label'] == 'negative')
        
        # Calculate net score (-1 to 1)
        return pos_score - neg_score
    except Exception as e:
        return 0.0 


# Applying the scoring function
news_df['Sentiment_Score'] = news_df['Text'].apply(compute_net_sentiment)

# Aggregate the scores by Ticker and Date to gather daily sentiment
daily_sentiment_df = news_df.groupby(['Ticker', 'Date'])['Sentiment_Score'].mean().reset_index()

# Format Date for merging
daily_sentiment_df['Date'] = pd.to_datetime(daily_sentiment_df['Date'])

print("\n--- New Sentiment Distribution ---")
print(daily_sentiment_df['Sentiment_Score'].describe())


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- New Sentiment Distribution ---
count    262.000000
mean       0.141306
std        0.529808
min       -0.967204
25%       -0.095085
50%        0.169136
75%        0.549962
max        0.943171
Name: Sentiment_Score, dtype: float64


Joining data

In [9]:
# Pack the dictionary
stock_dfs = {'MCHP': mchp, 'AMAT': amat, 'PCAR': pcar, 'CEG': ceg}

for ticker, df in stock_dfs.items():

    # Isolating the sentiment data
    ticker_sentiment = daily_sentiment_df[daily_sentiment_df['Ticker'] == ticker][['Date', 'Sentiment_Score']]
    
    # Set Date as index to perfectly align with stock data
    ticker_sentiment.set_index('Date', inplace=True)
    
    if 'Sentiment_Score' in df.columns:
        df = df.drop(columns=['Sentiment_Score'])

    # Left Join to Keep all stock market days
    stock_dfs[ticker] = df.join(ticker_sentiment, how='left')
    
    # Fill all NaN values with 0.0 for neutrality
    stock_dfs[ticker]['Sentiment_Score'] = stock_dfs[ticker]['Sentiment_Score'].fillna(0.0)
    
    print(f"{ticker} merge complete. Days without news filled with 0.0.")

# Reassigning back to working variables
mchp = stock_dfs['MCHP']
amat = stock_dfs['AMAT']
pcar = stock_dfs['PCAR']
ceg = stock_dfs['CEG']

print("\nVerify the merge (MCHP tail):")
print(mchp[['Close', 'Volume', 'Sentiment_Score']].tail())

MCHP merge complete. Days without news filled with 0.0.
AMAT merge complete. Days without news filled with 0.0.
PCAR merge complete. Days without news filled with 0.0.
CEG merge complete. Days without news filled with 0.0.

Verify the merge (MCHP tail):
                Close    Volume  Sentiment_Score
Date                                            
2026-04-13  73.550003   6724900         0.594852
2026-04-14  74.500000   7446400         0.439349
2026-04-15  74.489998   6869400         0.102791
2026-04-16  76.870003   7221100         0.599095
2026-04-17  78.760002  10894700         0.921237


# **Data preparation**

Creating target column:
This will allow the model to use todays data to predict tomorrow's price

In [51]:
for ticker, df in stock_dfs.items():

    # Shift the Close price up  by 1 row to create target column

    df['Target'] = df['Close'].shift(-1)
    df['Target_5d'] = df['Close'].shift(-5)

    # dropping N/A values
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)

Gathering all data in one DataFrame

In [11]:
# adding an identifier row to identify each stock entry

mchp['Ticker'] = 'MCHP'
ceg['Ticker'] = 'CEG'
pcar['Ticker'] = 'PCAR'
amat['Ticker'] = 'AMAT'

df_all = pd.concat([mchp, amat, pcar, ceg], axis=0)

df_all.insert(0, 'Ticker', df_all.pop('Ticker'))

Data split for XGboost

In [12]:
X_train_xgb, X_test_xgb = pd.DataFrame(), pd.DataFrame()
y_train_xgb, y_test_xgb = pd.Series(), pd.Series()

for ticker in ['MCHP', 'AMAT', 'PCAR', 'CEG']:
    
    ticker_df = df_all[df_all['Ticker'] == ticker]
    
    X = ticker_df.drop(columns=['Open', 'High', 'Low', 'Close', 'Target'])
    y = ticker_df['Target']
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.15, shuffle=False
    )
    
    X_train_xgb = pd.concat([X_train_xgb, X_train])
    X_test_xgb  = pd.concat([X_test_xgb,  X_test])
    y_train_xgb = pd.concat([y_train_xgb, y_train])
    y_test_xgb  = pd.concat([y_test_xgb,  y_test])

print(f"Train: {X_train_xgb.shape[0]} rows | Test: {X_test_xgb.shape[0]} rows")
print(f"\nTicker distribution in test set:")
print(X_test_xgb['Ticker'].value_counts())

Train: 2492 rows | Test: 440 rows

Ticker distribution in test set:
Ticker
MCHP    110
AMAT    110
PCAR    110
CEG     110
Name: count, dtype: int64


Generic Data Split

In [13]:
# separating values into X and y variables for training and testing
# Defining features in a variable 
features = list(mchp.drop(columns = ['Open', 'High', 'Low', 'Close', 'Target']).columns)

# dictionaries to hold 4 dataframes for each asset
X_dict = {}
y_dict = {}

# Loop to separate features from target column
for ticker, df in stock_dfs.items():

    # X gets the features, y gets the target
    X_dict[ticker] = df[features]
    y_dict[ticker] = df[['Target', 'Target_5d']]

# Initialising empty dictionaries to hold the splits
X_train_dict, X_test_dict = {}, {}
y_train_dict, y_test_dict = {}, {}

for ticker in ['MCHP', 'AMAT', 'PCAR', 'CEG']:
    X = X_dict[ticker]
    y = y_dict[ticker]
    
    # The Chronological Split 
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, shuffle=False
    )
    
    # Store the data back into dictionaries
    X_train_dict[ticker] = X_train
    X_test_dict[ticker] = X_test
    y_train_dict[ticker] = y_train
    y_test_dict[ticker] = y_test
    
    print(f"[{ticker}] Split Complete -> Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")

[MCHP] Split Complete -> Train: 586 rows | Test: 147 rows
[AMAT] Split Complete -> Train: 586 rows | Test: 147 rows
[PCAR] Split Complete -> Train: 586 rows | Test: 147 rows
[CEG] Split Complete -> Train: 586 rows | Test: 147 rows


In [14]:
selected_features = ['Volume', 'MACD', 'RSI_14', 'SD',
                     'Price_Momentum', 'Vol_Shock', 'Price_Range',
                     'Dist_from_EMA_20', 'Sentiment_Score',
                     'Close_Lag1', 'Close_Lag3', 'Close_Lag5',
                     'High_Lag1', 'Low_Lag1',
                     'Return_1d', 'Return_5d']

X_train_en = {ticker: df[selected_features] for ticker, df in X_train_dict.items()}
X_test_en  = {ticker: df[selected_features] for ticker, df in X_test_dict.items()}

In [62]:
# creating return target for LSTM only
# target = next-day return relative to the current close

y_train_lstm_target = {}
y_test_lstm_target = {}

for ticker in ['MCHP', 'AMAT', 'PCAR', 'CEG']:

    train_close = X_train_dict[ticker]['Close_Lag1'].values
    test_close = X_test_dict[ticker]['Close_Lag1'].values

    y_train_vals = y_train_dict[ticker]['Target'].values
    y_test_vals = y_test_dict[ticker]['Target'].values

    y_train_lstm_target[ticker] = pd.Series((y_train_vals - train_close) / train_close)
    y_test_lstm_target[ticker] = pd.Series((y_test_vals - test_close) / test_close)

    y_train_lstm_target[ticker].replace([np.inf, -np.inf], np.nan, inplace=True)
    y_test_lstm_target[ticker].replace([np.inf, -np.inf], np.nan, inplace=True)

    y_train_lstm_target[ticker].dropna(inplace=True)
    y_test_lstm_target[ticker].dropna(inplace=True)

In [63]:
# dictionaries for stock specific scalers

X_scalers = {}
y_scalers = {}

# empty dictionaries for scaled data

X_train_scaled_clean, X_test_scaled_clean = {}, {}
y_train_scaled, y_test_scaled = {}, {}

for ticker in ['MCHP', 'AMAT', 'PCAR', 'CEG']:

    # scaling input features
    # MinMax keeps features in a stable range for LSTM training
    X_scaler = MinMaxScaler()

    X_train_scaled_clean[ticker] = X_scaler.fit_transform(X_train_en[ticker])
    X_test_scaled_clean[ticker] = X_scaler.transform(X_test_en[ticker])

    # scaling return target
    # fit on training target only to avoid leakage
    y_scaler = StandardScaler()

    y_train_scaled[ticker] = y_scaler.fit_transform(
        y_train_lstm_target[ticker].values.reshape(-1, 1)
    )

    y_test_scaled[ticker] = y_scaler.transform(
        y_test_lstm_target[ticker].values.reshape(-1, 1)
    )

    # saving scalers for inverse transform later
    X_scalers[ticker] = X_scaler
    y_scalers[ticker] = y_scaler

Creating a reusable function to test error margins for models

In [17]:
def evaluator(dict_test, dict_pred, model_name):
    
    """Imports relevant metrics for testing machine learning models and runs mse, mae, rmse, mape, and r squared and returns
    the results as a dataframe"""

    from sklearn.metrics import mean_absolute_error
    from sklearn.metrics import root_mean_squared_error
    from sklearn.metrics import r2_score
    from sklearn.metrics import mean_squared_error
    from sklearn.metrics import mean_absolute_percentage_error
    print(f"Performance metrics for {model_name}")
    dict_metrics = {}
    for ticker in dict_test.keys():

        if ticker not in dict_pred:
            print(f"Warning: {ticker} missing from predictions. Skipping.")
            continue
            
        # Extract the variables once
        y_true = dict_test[ticker]
        y_pred = dict_pred[ticker]

        mae = mean_absolute_error(y_true, y_pred)
        rmse = root_mean_squared_error(y_true, y_pred)
        mse = mean_squared_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)
        mape = mean_absolute_percentage_error(y_true, y_pred) * 100

        dict_metrics[ticker] = {'MAE': mae, 'RMSE': rmse, 'MSE': mse, 
                                'R2': r2, 'MAPE': mape}
        
    metrics = pd.DataFrame(dict_metrics).T

    return metrics


# **Training ML models**

# ElasticNet

ElasticNet Training

In [18]:
elastic_net_models = {}

for key, value in X_train_en.items():
    
    # Instantiating the pipeline

    elastic_net_pipe = skpipe([
        ("Scaling", StandardScaler()), 
        ("ElasticNet", ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42))
    ])
    
    # Extract the target variable

    y_train = y_train_dict[key]
    
    # Fit dedicated pipeline

    elastic_net_pipe.fit(value, y_train)
    
    # Store the independent models

    elastic_net_models[key] = elastic_net_pipe

In [92]:
X_train_en['MCHP'].columns

Index(['Volume', 'MACD', 'RSI_14', 'SD', 'Price_Momentum', 'Vol_Shock',
       'Price_Range', 'Dist_from_EMA_20', 'Sentiment_Score', 'Close_Lag1',
       'Close_Lag3', 'Close_Lag5', 'High_Lag1', 'Low_Lag1', 'Return_1d',
       'Return_5d'],
      dtype='object')

Elastic net prediction testing

In [19]:
# elastic net prediction testing

elastic_net_test_pred = {}

for key, value in X_test_en.items():

    elastic_net_test_pred[key] = elastic_net_models[key].predict(value)

evaluator(y_test_dict, elastic_net_test_pred, "Elastic Net Regressor")

Performance metrics for Elastic Net Regressor


,MAE,RMSE,MSE,R2,MAPE
MCHP,1.440520,1.918931,3.682295,0.933382,2.194093
AMAT,9.406426,12.231454,149.608458,0.960108,3.138494
PCAR,1.552353,2.045097,4.182423,0.970201,1.392316
CEG,8.580842,11.235918,126.245864,0.902221,2.602710


ElasticNet Hyperparameter tuning

In [20]:
#  Hyper parameter tuning for elastic net models

tscv = TimeSeriesSplit(n_splits=5)

param_grid = {
    'ElasticNet__alpha':    [0.001, 0.01, 0.1, 1.0, 10.0],
    'ElasticNet__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
}

tuned_elastic_net_models = {}

print("Tuning Elastic Net — independent model per ticker...\n")

for ticker in ['MCHP', 'AMAT', 'PCAR', 'CEG']:

    pipe = skpipe([
        ('Scaling', StandardScaler()),
        ('ElasticNet', ElasticNet(random_state=42, max_iter=10000))
    ])

    grid_search = GridSearchCV(
        pipe,
        param_grid,
        cv=tscv,
        scoring='neg_mean_absolute_error',
        n_jobs=-1
    )

    grid_search.fit(X_train_en[ticker], y_train_dict[ticker])
    tuned_elastic_net_models[ticker] = grid_search.best_estimator_

    print(f"[{ticker}] Best alpha: {grid_search.best_params_['ElasticNet__alpha']} "
          f"| Best l1_ratio: {grid_search.best_params_['ElasticNet__l1_ratio']}")

# Predict with tuned models
tuned_en_pred = {ticker: tuned_elastic_net_models[ticker].predict(X_test_en[ticker])
                 for ticker in ['MCHP', 'AMAT', 'PCAR', 'CEG']}

# Evaluate
tuned_en_results = evaluator(y_test_dict, tuned_en_pred, "Tuned Elastic Net")
display(tuned_en_results)

Tuning Elastic Net — independent model per ticker...

[MCHP] Best alpha: 0.01 | Best l1_ratio: 1.0
[AMAT] Best alpha: 0.1 | Best l1_ratio: 1.0
[PCAR] Best alpha: 0.1 | Best l1_ratio: 1.0
[CEG] Best alpha: 0.1 | Best l1_ratio: 1.0
Performance metrics for Tuned Elastic Net


,MAE,RMSE,MSE,R2,MAPE
MCHP,1.307984,1.843716,3.399287,0.938502,1.989369
AMAT,8.213613,10.976477,120.483037,0.967874,2.776938
PCAR,1.442466,1.925172,3.706289,0.973593,1.296587
CEG,8.022945,10.695883,114.401913,0.911394,2.437737


Elastic Net 5 day prediction

In [21]:
y_train_en_5d = {}
y_test_en_5d = {}
for ticker in ['MCHP', 'AMAT', 'PCAR', 'CEG']:

    y_train_en_5d[ticker] = y_train_dict[ticker]['Target_5d']
    y_test_en_5d[ticker] = y_test_dict[ticker]['Target_5d']


In [22]:
# dictionaries for 5-day Elastic Net scalers
X_scalers_en_5d = {}
y_scalers_en_5d = {}

# empty dictionaries for scaled 5-day data
X_train_scaled_en_5d, X_test_scaled_en_5d = {}, {}
y_train_scaled_en_5d, y_test_scaled_en_5d = {}, {}

for ticker in ['MCHP', 'AMAT', 'PCAR', 'CEG']:

    # scaler for input features
    X_scaler = StandardScaler()

    X_train_scaled_en_5d[ticker] = X_scaler.fit_transform(X_train_en[ticker])
    X_test_scaled_en_5d[ticker] = X_scaler.transform(X_test_en[ticker])

    # scaler for 5-day target
    y_scaler = StandardScaler()

    y_train_scaled_en_5d[ticker] = y_scaler.fit_transform(
        y_train_en_5d[ticker].values.reshape(-1, 1)
    )

    y_test_scaled_en_5d[ticker] = y_scaler.transform(
        y_test_en_5d[ticker].values.reshape(-1, 1)
    )

    # store scalers
    X_scalers_en_5d[ticker] = X_scaler
    y_scalers_en_5d[ticker] = y_scaler

    print(f"{ticker} scaling complete")

MCHP scaling complete
AMAT scaling complete
PCAR scaling complete
CEG scaling complete


In [23]:
# dictionaries to store trained 5-day Elastic Net models
elastic_models_5d = {}

# training one 5-day Elastic Net model per stock
for ticker in ['MCHP', 'AMAT', 'PCAR', 'CEG']:

    print(f"\nTraining 5-day Elastic Net for {ticker}...")

    # using same baseline setup as the original Elastic Net
    model = ElasticNet(
        alpha=0.1,
        l1_ratio=0.5,
        random_state=42
    )

    model.fit(
        X_train_scaled_en_5d[ticker],
        y_train_scaled_en_5d[ticker].ravel()
    )

    # store trained model
    elastic_models_5d[ticker] = model

    print(f"{ticker} training complete")
    print("-" * 60)


Training 5-day Elastic Net for MCHP...
MCHP training complete
------------------------------------------------------------

Training 5-day Elastic Net for AMAT...
AMAT training complete
------------------------------------------------------------

Training 5-day Elastic Net for PCAR...
PCAR training complete
------------------------------------------------------------

Training 5-day Elastic Net for CEG...
CEG training complete
------------------------------------------------------------


In [24]:
# dictionaries to store 5-day Elastic Net predictions
elastic_pred_scaled_5d = {}
elastic_pred_raw_5d = {}

# testing one 5-day Elastic Net model per stock
for ticker in ['MCHP', 'AMAT', 'PCAR', 'CEG']:

    print(f"Testing 5-day Elastic Net for {ticker}...")

    # predict on scaled test features
    y_pred_scaled = elastic_models_5d[ticker].predict(X_test_scaled_en_5d[ticker])

    # reshape for inverse transform
    y_pred_scaled = y_pred_scaled.reshape(-1, 1)

    # convert back to raw 5-day closing price
    y_pred_raw = y_scalers_en_5d[ticker].inverse_transform(y_pred_scaled).flatten()

    # store predictions
    elastic_pred_scaled_5d[ticker] = y_pred_scaled
    elastic_pred_raw_5d[ticker] = y_pred_raw

    print(f"{ticker} testing complete")

# evaluate 5-day Elastic Net predictions
elastic_results_5d = evaluator(y_test_en_5d, elastic_pred_raw_5d, "Elastic Net 5D")
display(elastic_results_5d)

Testing 5-day Elastic Net for MCHP...
MCHP testing complete
Testing 5-day Elastic Net for AMAT...
AMAT testing complete
Testing 5-day Elastic Net for PCAR...
PCAR testing complete
Testing 5-day Elastic Net for CEG...
CEG testing complete
Performance metrics for Elastic Net 5D


,MAE,RMSE,MSE,R2,MAPE
MCHP,1.677844,2.197453,4.828799,0.912640,2.576056
AMAT,13.833906,17.169839,294.803371,0.921393,4.538303
PCAR,2.247889,2.858617,8.171688,0.941778,1.981644
CEG,11.940166,14.570346,212.294992,0.835574,3.599423


# XGboost 

XGboost Training

In [25]:
# Only encode the Ticker column, pass all numeric columns through untouched
preprocessor = ColumnTransformer(transformers=[
    ('ticker_encoder', OrdinalEncoder(), ['Ticker'])
],  remainder='passthrough')  # passthrough leaves all other columns unchanged

xgb_pipeline = skpipe([
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.5,
        random_state=42,
        n_jobs=1
    ))
])

# Train
xgb_pipeline.fit(X_train_xgb, y_train_xgb)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ticker_encoder',
                                                  OrdinalEncoder(),
                                                  ['Ticker'])])),
                ('model',
                 XGBRegressor(base_score=None, booster=None, callbacks=None,
                              colsample_bylevel=None, colsample_bynode=None,
                              colsample_bytree=0.8, device=None,
                              early_stopping_rounds=None,
                              enable_categorical=False, eval_metric=None,
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.05,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=3, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=400, n_jobs=1,
                              num_parallel_tree=None, ...))])

XGboost prediction testing

In [26]:
# Predict
xgb_pipeline_pred = xgb_pipeline.predict(X_test_xgb)

# Evaluate per ticker
results = {}

for ticker in ['MCHP', 'AMAT', 'PCAR', 'CEG']:
    mask   = X_test_xgb['Ticker'] == ticker
    y_true = y_test_xgb[mask]
    y_pred = xgb_pipeline_pred[mask]
    
    results[ticker] = {
        'MAE':  mean_absolute_error(y_true, y_pred),
        'RMSE': root_mean_squared_error(y_true, y_pred),
        'MSE':  mean_squared_error(y_true, y_pred),
        'R2':   r2_score(y_true, y_pred),
        'MAPE': mean_absolute_percentage_error(y_true, y_pred) * 100
    }

xgb_results_df = pd.DataFrame(results).T
print("XGBoost Pipeline Results")
display(xgb_results_df)

XGBoost Pipeline Results


,MAE,RMSE,MSE,R2,MAPE
MCHP,0.523162,0.773640,0.598519,0.991239,0.814427
AMAT,3.466592,4.889627,23.908455,0.991119,1.035245
PCAR,0.764859,1.054322,1.111595,0.987582,0.648918
CEG,3.149852,3.977431,15.819961,0.985446,0.989149


# LSTM

Defining an LSTM function

In [64]:
# setting seeds for better reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# defining a simple LSTM model for reproducibility
def build_lstm_model(n_features, seq_length):

    model = Sequential()

    # single LSTM layer to learn short sequence patterns
    model.add(LSTM(16, input_shape=(seq_length, n_features)))

    # dropout helps reduce overfitting
    model.add(Dropout(0.2))

    # small dense layer for extra learning capacity
    model.add(Dense(8, activation='relu'))

    # final output layer predicts next-day return
    model.add(Dense(1))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='mse'
    )

    return model

Creating an LSTM sequencer function

In [65]:
# reusable sequence creater

def sequencer(X, y, seq_length):
    """Converts flat 2D arrays into 3D sequences for LSTM"""

    # variables to hold sequences

    X_seq, y_seq = [], []

    for i in range(seq_length, len(X)):
        X_seq.append(X[i-seq_length:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)

Preparing data specifically for LSTM

In [66]:
# Defining sequence for each input block
sequence = 10

X_train_lstm, X_test_lstm = {}, {}
y_train_lstm, y_test_lstm = {}, {}

for ticker in ['MCHP', 'AMAT', 'PCAR', 'CEG']:

    X_train_lstm[ticker], y_train_lstm[ticker] = sequencer(
        X_train_scaled_clean[ticker],
        y_train_scaled[ticker],
        sequence
    )

    X_test_lstm[ticker], y_test_lstm[ticker] = sequencer(
        X_test_scaled_clean[ticker],
        y_test_scaled[ticker],
        sequence
    )

    print(f"[{ticker}] X_train_seq: {X_train_lstm[ticker].shape} | "
          f"X_test_seq: {X_test_lstm[ticker].shape}")

[MCHP] X_train_seq: (576, 10, 16) | X_test_seq: (137, 10, 16)
[AMAT] X_train_seq: (576, 10, 16) | X_test_seq: (137, 10, 16)
[PCAR] X_train_seq: (576, 10, 16) | X_test_seq: (137, 10, 16)
[CEG] X_train_seq: (576, 10, 16) | X_test_seq: (137, 10, 16)


LSTM validation

In [67]:
# splitting training sequences into train and validation

X_train_final_lstm = {}
X_val_lstm = {}
y_train_final_lstm = {}
y_val_lstm = {}

val_size = 0.15

for ticker in ['MCHP', 'AMAT', 'PCAR', 'CEG']:

    split_index = int(len(X_train_lstm[ticker]) * (1 - val_size))

    X_train_final_lstm[ticker] = X_train_lstm[ticker][:split_index]
    X_val_lstm[ticker] = X_train_lstm[ticker][split_index:]

    y_train_final_lstm[ticker] = y_train_lstm[ticker][:split_index]
    y_val_lstm[ticker] = y_train_lstm[ticker][split_index:]

    print(f"{ticker}")
    print("X_train_final shape:", X_train_final_lstm[ticker].shape)
    print("X_val shape:", X_val_lstm[ticker].shape)
    print("y_train_final shape:", y_train_final_lstm[ticker].shape)
    print("y_val shape:", y_val_lstm[ticker].shape)
    print("-" * 50)

MCHP
X_train_final shape: (489, 10, 16)
X_val shape: (87, 10, 16)
y_train_final shape: (489, 1)
y_val shape: (87, 1)
--------------------------------------------------
AMAT
X_train_final shape: (489, 10, 16)
X_val shape: (87, 10, 16)
y_train_final shape: (489, 1)
y_val shape: (87, 1)
--------------------------------------------------
PCAR
X_train_final shape: (489, 10, 16)
X_val shape: (87, 10, 16)
y_train_final shape: (489, 1)
y_val shape: (87, 1)
--------------------------------------------------
CEG
X_train_final shape: (489, 10, 16)
X_val shape: (87, 10, 16)
y_train_final shape: (489, 1)
y_val shape: (87, 1)
--------------------------------------------------


LSTM training

In [68]:
# dictionaries to store trained models, history, and predictions
lstm_models = {}
lstm_histories = {}

# early stopping stops training when validation loss stops improving
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=25,
    restore_best_weights=True
)

# training one LSTM model per stock
for ticker in ['MCHP', 'AMAT', 'PCAR', 'CEG']:

    print(f"\nTraining LSTM for {ticker}...")

    model = build_lstm_model(
        n_features=X_train_final_lstm[ticker].shape[2],
        seq_length=X_train_final_lstm[ticker].shape[1]
    )

    history = model.fit(
        X_train_final_lstm[ticker],
        y_train_final_lstm[ticker],
        validation_data=(X_val_lstm[ticker], y_val_lstm[ticker]),
        epochs=200,
        batch_size=16,
        callbacks=[early_stop],
        verbose=1
    )

    # store trained model and history
    lstm_models[ticker] = model
    lstm_histories[ticker] = history

    best_val_loss = min(history.history['val_loss'])
    print(f"{ticker} training complete")
    print(f"Best validation loss: {best_val_loss:.6f}")
    print("-" * 60)


Training LSTM for MCHP...
Epoch 1/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 1.0501 - val_loss: 0.8266
Epoch 2/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.0312 - val_loss: 0.8223
Epoch 3/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.0299 - val_loss: 0.8177
Epoch 4/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.0263 - val_loss: 0.8152
Epoch 5/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.0284 - val_loss: 0.8144
Epoch 6/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.0231 - val_loss: 0.8107
Epoch 7/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.0169 - val_loss: 0.8112
Epoch 8/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.0206 - val_loss: 0.8054
Epoch 9/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.0206 - val_loss: 0.8136
Epoch 10/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.0294 - val_loss: 0.8062
Epoch 11/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.0194 - val_loss: 0.7984
Epoch 12/200
31/31 ━━━━━━━━━━━━━━━━━

LSTM predictions

In [69]:
# dictionaries to store LSTM predictions
lstm_pred_scaled = {}
lstm_pred_raw = {}

# making predictions for each stock
for ticker in ['MCHP', 'AMAT', 'PCAR', 'CEG']:

    print(f"Testing LSTM for {ticker}...")

    # predict scaled returns
    y_pred_scaled = lstm_models[ticker].predict(X_test_lstm[ticker], verbose=0)

    # convert scaled returns back to return values
    y_pred_return = y_scalers[ticker].inverse_transform(y_pred_scaled).flatten()

    # base close price aligned with sequence shift
    base_close = X_test_dict[ticker]['Close_Lag1'].iloc[sequence:].values

    # convert predicted return back to raw next-day price
    y_pred_raw = base_close * (1 + y_pred_return)

    # store predictions
    lstm_pred_scaled[ticker] = y_pred_scaled
    lstm_pred_raw[ticker] = y_pred_raw

Testing LSTM for MCHP...
Testing LSTM for AMAT...
Testing LSTM for PCAR...
Testing LSTM for CEG...


Testing LSTM predictions

In [71]:
# aligning actual raw prices with LSTM predictions

y_test_lstm_raw = {}

for ticker in ['MCHP', 'AMAT', 'PCAR', 'CEG']:

    actual_return = y_test_lstm_target[ticker].iloc[sequence:].values
    base_close = X_test_dict[ticker]['Close_Lag1'].iloc[sequence:].values

    y_test_lstm_raw[ticker] = base_close * (1 + actual_return)
# evaluating LSTM predictions against raw closing prices

lstm_results = evaluator(y_test_lstm_raw, lstm_pred_raw, "LSTM")
display(lstm_results)

Performance metrics for LSTM


,MAE,RMSE,MSE,R2,MAPE
MCHP,1.959019,2.642884,6.984834,0.881270,2.967564
AMAT,9.569855,13.094549,171.467216,0.949063,3.170337
PCAR,1.936449,2.448285,5.994098,0.955771,1.729939
CEG,10.710389,13.995748,195.880972,0.858185,3.291516


Scaling data for MLP pipeline

In [82]:
# dictionaries for stock specific scalers
X_scalers_mlp = {}
y_scalers_mlp = {}

# empty dictionaries for scaled data
X_train_scaled_mlp, X_test_scaled_mlp = {}, {}
y_train_scaled_mlp, y_test_scaled_mlp = {}, {}

# scaling data separately for each stock
for ticker in ['MCHP', 'AMAT', 'PCAR', 'CEG']:

    # scaler for input features
    X_scaler = StandardScaler()

    X_train_scaled_mlp[ticker] = X_scaler.fit_transform(X_train_en[ticker])
    X_test_scaled_mlp[ticker] = X_scaler.transform(X_test_en[ticker])

    # scaler for target variable
    y_scaler = StandardScaler()

    y_train_scaled_mlp[ticker] = y_scaler.fit_transform(
        y_train_dict[ticker]['Target'].values.reshape(-1, 1)
    )
  
    y_test_scaled_mlp[ticker] = y_scaler.transform(
        y_test_dict[ticker]['Target'].values.reshape(-1, 1)
    )

    # storing scalers for inverse transform later
    X_scalers_mlp[ticker] = X_scaler
    y_scalers_mlp[ticker] = y_scaler

# MLP

defining MLP model

In [73]:
mlp_model = MLPRegressor(
    hidden_layer_sizes = (128, 64), activation = 'relu', alpha = 0.001, batch_size = 16, 
    learning_rate_init = 0.001, max_iter = 1000, early_stopping = True, 
    validation_fraction=0.15, n_iter_no_change=15, random_state=42)

MLP model training

In [84]:
# dictionaries to store trained models and training results
mlp_models = {}

# training one MLP model per stock
for ticker in ['MCHP', 'AMAT', 'PCAR', 'CEG']:

    print(f"\nTraining MLP for {ticker}...")

    # train model using scaled data
    mlp_model.fit(
        X_train_scaled_mlp[ticker],
        y_train_scaled_mlp[ticker].ravel()   # flatten target for sklearn
    )

    # store trained model
    mlp_models[ticker] = mlp_model

    print(f"{ticker} training complete")
    print(f"Iterations: {mlp_model.n_iter_}")
    print("-" * 60)


Training MLP for MCHP...
MCHP training complete
Iterations: 31
------------------------------------------------------------

Training MLP for AMAT...
AMAT training complete
Iterations: 38
------------------------------------------------------------

Training MLP for PCAR...
PCAR training complete
Iterations: 20
------------------------------------------------------------

Training MLP for CEG...
CEG training complete
Iterations: 23
------------------------------------------------------------


Testing MLP model

In [87]:
# dictionaries to store predictions
mlp_pred_scaled = {}
mlp_pred_raw = {}

# testing one MLP per stock
for ticker in ['MCHP', 'AMAT', 'PCAR', 'CEG']:

    print(f"Testing MLP for {ticker}...")

    # predict on scaled data
    y_pred_scaled = mlp_models[ticker].predict(X_test_scaled_mlp[ticker])

    # reshape for inverse transform
    y_pred_scaled = y_pred_scaled.reshape(-1, 1)

    # convert back to raw closing price
    y_pred_raw = y_scalers_mlp[ticker].inverse_transform(y_pred_scaled).flatten()

    # store predictions
    mlp_pred_scaled[ticker] = y_pred_scaled
    mlp_pred_raw[ticker] = y_pred_raw


y_test_single = {}

for ticker in ['MCHP', 'AMAT', 'PCAR', 'CEG']:
    y_test_single[ticker] = y_test_dict[ticker]['Target'].values

mlp_results = evaluator(y_test_single, mlp_pred_raw, "MLP")
display(mlp_results)

Testing MLP for MCHP...
Testing MLP for AMAT...
Testing MLP for PCAR...
Testing MLP for CEG...
Performance metrics for MLP


,MAE,RMSE,MSE,R2,MAPE
MCHP,2.023680,2.810299,7.897778,0.857117,3.137948
AMAT,14.327682,17.863617,319.108811,0.914913,4.596009
PCAR,1.898560,2.509825,6.299223,0.955119,1.682528
CEG,10.643302,13.700463,187.702679,0.854621,3.240808


MLP hypertuning

Parameter grid

In [90]:
mlp_param_dist = {
    'hidden_layer_sizes': [
        (32,),
        (64,),
        (32, 16),
        (64, 32),
        (128, 64)
    ],
    'activation': ['relu', 'tanh'],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate_init': [0.0005, 0.001, 0.005],
    'batch_size': [16, 32],
    'max_iter': [500, 750]
}

# time series cross validation
tscv = TimeSeriesSplit(n_splits=3)

Hyperparameter tuning

In [91]:
# dictionaries to store tuned models and search results
mlp_tuned_models = {}
mlp_search_results = {}

# tune one MLP per stock using time series split
for ticker in ['MCHP', 'AMAT', 'PCAR', 'CEG']:

    print(f"\nTuning MLP for {ticker}...")

    base_model = MLPRegressor(
        solver='adam',
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=15,
        random_state=42
    )

    search = RandomizedSearchCV(
        estimator=base_model,
        param_distributions=mlp_param_dist,
        n_iter=12,
        scoring='r2',
        cv=tscv,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    search.fit(
        X_train_scaled_mlp[ticker],
        y_train_scaled_mlp[ticker].ravel()
    )

    mlp_tuned_models[ticker] = search.best_estimator_
    mlp_search_results[ticker] = search

    print(f"{ticker} tuning complete")
    print("Best params:", search.best_params_)
    print("Best CV R2:", search.best_score_)
    print("-" * 60)


Tuning MLP for MCHP...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
MCHP tuning complete
Best params: {'max_iter': 500, 'learning_rate_init': 0.001, 'hidden_layer_sizes': (32,), 'batch_size': 16, 'alpha': 0.01, 'activation': 'tanh'}
Best CV R2: 0.8228208053696414
------------------------------------------------------------

Tuning MLP for AMAT...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
AMAT tuning complete
Best params: {'max_iter': 500, 'learning_rate_init': 0.001, 'hidden_layer_sizes': (64,), 'batch_size': 16, 'alpha': 0.001, 'activation': 'tanh'}
Best CV R2: 0.8661327969286902
------------------------------------------------------------

Tuning MLP for PCAR...
Fitting 3 folds for each of 12 candidates, totalling 36 fits


/opt/anaconda3/envs/COM731/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


PCAR tuning complete
Best params: {'max_iter': 500, 'learning_rate_init': 0.001, 'hidden_layer_sizes': (32,), 'batch_size': 16, 'alpha': 0.01, 'activation': 'tanh'}
Best CV R2: 0.859656897153851
------------------------------------------------------------

Tuning MLP for CEG...
Fitting 3 folds for each of 12 candidates, totalling 36 fits


/opt/anaconda3/envs/COM731/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


CEG tuning complete
Best params: {'max_iter': 500, 'learning_rate_init': 0.0005, 'hidden_layer_sizes': (128, 64), 'batch_size': 32, 'alpha': 0.01, 'activation': 'tanh'}
Best CV R2: 0.8898943765227799
------------------------------------------------------------


Didn't work